# Interactive web maps with `InteractiveMap`

`digitalearth.interactive` is the **interactive-2D web-map tier**: the same pyramids data and auto-style
pipeline as the static `Map`, rendered through the **HoloViz stack** (GeoViews → HoloViews → Bokeh, plus
Datashader and Panel) so the result pans, zooms and hovers in the browser.

It is a *renderer, not a GIS engine*: every layer is built from pyramids-sourced arrays / GeoDataFrames and
all CRS work happens upstream in pyramids — never `xarray`/`rasterio`/`cartopy`. The engine import is **lazy**,
so `import digitalearth.interactive` works without the `interactive` extra; only a builder call needs it.

These notebooks use the bundled **Rhine-basin hydrology** data (gauges, river network, basin) and a
global **WorldClim temperature** stack — all real files under `examples/data/`.

**Setup** — find the repo root and load the Bokeh extension. Every notebook starts like this.

In [ ]:
from pathlib import Path

# Resolve the repo root so the bundled sample data is found whether this runs from
# docs/examples/interactive/ (mkdocs) or the repository root.
ROOT = Path.cwd()
while not (ROOT / "examples" / "data" / "LisbonElevation.tif").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "examples" / "data"

import holoviews as hv
hv.extension("bokeh")            # the interactive tier renders through Bokeh

One call, one finished map: `InteractiveMap().image(...)`. The cell's last expression *is* the map — displaying it renders the interactive Bokeh plot (pan/zoom/hover with the toolbar on the right).

In [ ]:
from pyramids.dataset import Dataset
from digitalearth.interactive import InteractiveMap

dem = Dataset.read_file(str(DATA / "LisbonElevation.tif"))
m = InteractiveMap(crs=dem.epsg, title="Lisbon elevation")
m.image(dem, cmap="terrain")
m

The builders **chain** (each returns the map) and compose as overlays. Below: a contour set on top of the shaded field, with a colorbar. Try the wheel-zoom and box-zoom tools.

In [ ]:
m = InteractiveMap(crs=dem.epsg, title="elevation + contours")
m.image(dem, cmap="terrain").contours(dem, levels=8, color="black")
m